In [ ]:
# import os
# import requests

# if not os.path.exists("the-verdict.txt"):
#     url = (
#         "https://raw.githubusercontent.com/rasbt/"
#         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
#         "the-verdict.txt"
#     )
#     file_path = "the-verdict.txt"

#     response = requests.get(url, timeout=30)
#     response.raise_for_status()
#     with open(file_path, "wb") as f:
#         f.write(response.content)

In [2]:
with open("data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [5]:
import re

text = "Hello, my dog is cute"
result = re.split(r"\s", text)

print(result)

['Hello,', 'my', 'dog', 'is', 'cute']


In [6]:

result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'my', ' ', 'dog', ' ', 'is', ' ', 'cute']


In [7]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'my', 'dog', 'is', 'cute']


In [11]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
print(result[:10])
print("Total number of tokens:", len(result))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']
Total number of tokens: 4690


In [49]:
unique_words = set(sorted(result))
voacabulary_size = len(unique_words)
print("Total number of unique tokens:", voacabulary_size)

Total number of unique tokens: 1130


In [50]:
vocab = {token: idx for idx, token in enumerate(unique_words)}
print("Vocabulary size:", len(vocab))
print("First 5 tokens in the vocabulary:\n", list(vocab.items())[:5])

Vocabulary size: 1130
First 5 tokens in the vocabulary:
 [('forehead', 0), ('long', 1), ('ironic', 2), ('luncheon-table', 3), ('itself', 4)]


In [51]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        return [self.vocab[token] for token in tokens if token in self.vocab]

    def decode(self, token_ids):
        text = " ".join([self.inv_vocab[i] for i in token_ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [52]:
tokenizer = SimpleTokenizer(vocab)

sample_text = "Hello, my dog is cute"
encoded = tokenizer.encode(sample_text)
print("Encoded:", encoded)

decoded = tokenizer.decode(encoded)
print("Decoded:", decoded)

Encoded: [375, 374, 61]
Decoded: , my is


In [53]:
text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
           
encoded = tokenizer.encode(text)
print("Encoded:", encoded)  

decoded = tokenizer.decode(encoded)
print("Decoded:", decoded)

Encoded: [594, 1017, 390, 781, 86, 63, 965, 530, 375, 306, 743, 375, 594, 866, 146, 831, 695, 138, 288, 309, 146]
Decoded: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [57]:
all_tokens = sorted(list(set(result)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

print("Vocabulary size:", len(vocab))
print("First 5 tokens in the vocabulary:\n", list(vocab.items())[-5:])

Vocabulary size: 1132
First 5 tokens in the vocabulary:
 [('younger', 1127), ('your', 1128), ('yourself', 1129), ('<|endoftext|>', 1130), ('<|unk|>', 1131)]


In [58]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        return [self.vocab.get(token, self.vocab["<|unk|>"]) for token in tokens]
    
    def decode(self, token_ids):
        text = " ".join([self.inv_vocab.get(i, "<|unk|>") for i in token_ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [59]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [61]:
# byte-pair encoding (BPE) 

import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [62]:
tokenizer.encode("Hello, my dog is cute")

[15496, 11, 616, 3290, 318, 13779]

In [64]:
tokenizer.decode([15496, 11, 616, 3290, 318, 13779])

'Hello, my dog is cute'

In [65]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
